# Music Annotation Project - Notebook 85
## HITL Annotation-Probability Heatmaps and Band-Expansion Atlas

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook is a read-only diagnostic. It does not retrain, refit, recalibrate, modify reviewed masks, or promote a policy.

### Scientific question
Within A1-preserved `AMBIGUOUS` pixels on complete reviewed HITL pages, where does the original YOLO annotation-confidence mass lie, and how much additional material would move from exact observed-byte preservation to the frozen local paper estimate if the high-confidence annotation band were expanded?

### Important distinction
- `p_annotation` is the original detector evidence saved by the HITL generator.
- `p_print` is the observed-print evidence saved by a completed whole-page V8/V8.9/A1/A3 run where available.
- A high `p_annotation` value does not by itself prove that removal is safe. Candidate maps are therefore diagnostic and remain constrained to `AMBIGUOUS` inside reviewed authority.

### Visual scheme
Probability bands use a high-contrast rainbow palette rather than shades of red:

| Band | Colour |
|---|---|
| >= 0.995 | violet |
| 0.990-0.995 | magenta |
| 0.950-0.990 | red-orange |
| 0.900-0.950 | orange |
| 0.800-0.900 | yellow |
| 0.600-0.800 | green |
| 0.400-0.600 | cyan |
| 0.200-0.400 | blue |
| < 0.200 | grey |

The notebook creates:
- full-page annotation-probability heatmaps;
- rainbow band overlays within A1 `AMBIGUOUS`;
- incremental candidate-to-paper overlays at expanding annotation thresholds;
- optional two-axis `p_annotation x p_print` overlays;
- counterfactual A1-plus-paper pages;
- page/band/component tables and ranked contact sheets;
- a curated final upload bundle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform, math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_DIR=ROOT/'source'/'annotated_only'
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
NOTEBOOK83_ROOT=ROOT/'hitl_v89_a1_a3_comparison'
V89_OPERATOR_OVERRIDE=None
NOTEBOOK83_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'hitl_probability_band_audit_notebook85'
PAGE_ALLOWLIST=None
MAX_PAGES=None
MASK_DILATION=2
OVERLAY_ALPHA=.58
REVIEW_RADIUS=1
HEATMAP_MIN=.05
CONTACT_SHEET_PAGES=20
PAPER_THRESHOLDS=[.995,.990,.975,.950,.925,.900,.850,.800,.700,.600,.500]
PPRINT_CEILINGS=[.10,.25,.50]
BANDS=[
 {'id':'A_GE_0995','low':.995,'high':1.000001,'label':'>= 0.995','bgr':(255,0,180)},
 {'id':'B_099_0995','low':.990,'high':.995,'label':'0.990-0.995','bgr':(255,0,255)},
 {'id':'C_095_099','low':.950,'high':.990,'label':'0.950-0.990','bgr':(0,70,255)},
 {'id':'D_090_095','low':.900,'high':.950,'label':'0.900-0.950','bgr':(0,165,255)},
 {'id':'E_080_090','low':.800,'high':.900,'label':'0.800-0.900','bgr':(0,255,255)},
 {'id':'F_060_080','low':.600,'high':.800,'label':'0.600-0.800','bgr':(0,210,0)},
 {'id':'G_040_060','low':.400,'high':.600,'label':'0.400-0.600','bgr':(255,255,0)},
 {'id':'H_020_040','low':.200,'high':.400,'label':'0.200-0.400','bgr':(255,90,0)},
 {'id':'I_LT_020','low':0.0,'high':.200,'label':'< 0.200','bgr':(120,120,120)},
]
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
RUN=OUTPUT_ROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook85')
TABLES=RUN/'tables'; WHOLE=RUN/'whole_pages'; VISUALS=RUN/'visual_audit'; AUDIT=RUN/'audit'; UPLOAD=RUN/'upload'
for p in [RUN,TABLES,WHOLE,VISUALS,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)

def require(x,msg):
 if not x: raise RuntimeError(msg)
def md5_file(p):
 h=hashlib.md5()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def read_gray(p):
 x=cv2.imread(str(p),0); require(x is not None,f'Unreadable image: {p}'); return x
def image_files(folder):return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if Path(folder).exists() else []
def latest_complete(root,required):
 hits=[]
 if Path(root).exists():
  for p in Path(root).rglob('*'):
   if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
print('Output:',RUN)

## 2. Resolve complete reviewed HITL packages, original confidence maps and Notebook 83 page artefacts

In [ ]:
def read_audits(folder,label):
 records=[]; failures=[]
 for p in sorted(Path(folder).glob('*.json')):
  try:
   v=json.loads(p.read_text()); require(isinstance(v,dict),f'{p}: root not object');v['_audit_path']=str(p);v['_audit_type']=label;records.append(v)
  except Exception as e:failures.append({'audit_type':label,'audit_path':str(p),'error':str(e)})
 return records,failures
reviews,rf=read_audits(REVIEW_AUDIT_DIR,'review');drafts,df=read_audits(DRAFT_AUDIT_DIR,'draft')
if rf+df:
 pd.DataFrame(rf+df).to_csv(AUDIT/'audit_parse_failures.csv',index=False);raise RuntimeError('Unreadable HITL audit JSON. See audit_parse_failures.csv')
def index_unique(records,key,label):
 out={};dups=[]
 for r in records:
  k=str(r.get(key,'')).lower()
  if not k:continue
  if k in out:dups.append({'type':label,'key':k,'first':out[k]['_audit_path'],'second':r['_audit_path']})
  else:out[k]=r
 if dups:
  pd.DataFrame(dups).to_csv(AUDIT/f'duplicate_{label}.csv',index=False);raise RuntimeError(f'Duplicate {label} audit identity')
 return out
review_by=index_unique(reviews,'source_md5','review');draft_by=index_unique(drafts,'source_md5','draft')
NB83=Path(NOTEBOOK83_RUN_OVERRIDE) if NOTEBOOK83_RUN_OVERRIDE else latest_complete(NOTEBOOK83_ROOT,['tables/hitl_reviewed_roster.csv'])
nb83_roster=pd.read_csv(NB83/'tables'/'hitl_reviewed_roster.csv',dtype=str).fillna('') if NB83 else pd.DataFrame()
nb83_by={str(r.page_key):r for _,r in nb83_roster.iterrows()} if len(nb83_roster) else {}
rows=[]; exclusions=[]
for source in image_files(SOURCE_DIR):
 if PAGE_ALLOWLIST is not None and source.stem not in set(PAGE_ALLOWLIST):continue
 smd5=md5_file(source).lower(); rev=review_by.get(smd5); draft=draft_by.get(smd5)
 def skip(reason):exclusions.append({'page_key':source.stem,'source_file':str(source),'source_md5':smd5,'reason':reason})
 if rev is None:skip('no_review_audit');continue
 if draft is None:skip('no_draft_audit_or_confidence_map');continue
 vals={k:rev.get(k) for k in ['aligned_output','corrected_mask_output']};vals['confidence_map']=draft.get('confidence_map')
 if any(not v for v in vals.values()):skip('incomplete_review_or_draft_package');continue
 paths={k:Path(v) for k,v in vals.items()}
 if any(not p.exists() for p in paths.values()):skip('referenced_file_missing');continue
 obs=read_gray(paths['aligned_output']);seed=read_gray(paths['corrected_mask_output']);conf=np.load(paths['confidence_map']).astype(np.float32)
 if obs.shape!=seed.shape or obs.shape!=conf.shape:skip('shape_mismatch');continue
 # Optional artefacts from Notebook 83. Resolve by explicit paths then conventional page folder.
 nrow=nb83_by.get(source.stem); a1_path='';pprint_path='';amb_path=''
 candidates=[]
 if nrow is not None:
  for c in ['page_output_dir','output_dir','page_dir']:
   if c in nb83_roster.columns and str(nrow.get(c,'')):candidates.append(Path(str(nrow[c])))
 candidates += [NB83/'pages'/source.stem if NB83 else Path('/nonexistent'),NB83/'whole_pages'/source.stem if NB83 else Path('/nonexistent')]
 for d in candidates:
  for name in ['a1.png','A1.png']:
   if (d/name).exists():a1_path=str(d/name)
  for name in ['p_print.npy','pprint.npy','p_print.npz']:
   if (d/name).exists():pprint_path=str(d/name)
  for name in ['ambiguous.png','g80_ambiguous.png']:
   if (d/name).exists():amb_path=str(d/name)
 rows.append({'page_key':source.stem,'source_file':str(source),'source_md5':smd5,'observed_path':str(paths['aligned_output']),'seed_path':str(paths['corrected_mask_output']),'confidence_path':str(paths['confidence_map']),'review_audit':rev['_audit_path'],'draft_audit':draft['_audit_path'],'a1_path':a1_path,'p_print_path':pprint_path,'ambiguous_path':amb_path})
roster=pd.DataFrame(rows); exclusions=pd.DataFrame(exclusions)
if MAX_PAGES is not None:roster=roster.head(MAX_PAGES)
require(len(roster)>0,'No complete reviewed HITL packages with confidence maps.')
roster.to_csv(TABLES/'notebook85_input_roster.csv',index=False);exclusions.to_csv(TABLES/'notebook85_input_exclusions.csv',index=False)
print('Included:',len(roster),'Excluded:',len(exclusions),'Notebook83:',NB83)

## 3. Frozen V7.8 helpers and optional V8.9 operator resolution

In [ ]:
MASK_DILATION=2;CROP_MARGIN=48;GEOMETRY_THRESHOLD=80;BAND_WIDTH=4
def binary(mask):return np.where(mask>0,255,0).astype(np.uint8)
def prepare_mask(seed,dilation=MASK_DILATION):
 m=binary(seed)
 if dilation>0:m=cv2.dilate(m,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
 return m
def padded_components(mask):
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;boxes=[]
 for i in range(1,n):
  x,y,bw,bh,a=stats[i]
  if a>=8:boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
 changed=True
 while changed:
  changed=False;merged=[]
  while boxes:
   ax0,ay0,ax1,ay1=boxes.pop();rest=[]
   for bx0,by0,bx1,by1 in boxes:
    if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
    else:rest.append((bx0,by0,bx1,by1))
   boxes=rest;merged.append((ax0,ay0,ax1,ay1))
  boxes=merged
 return boxes
def telea_reconstruct(observed,mask):
 out=observed.copy()
 for x0,y0,x1,y1 in padded_components(mask):
  im=observed[y0:y1,x0:x1];mk=mask[y0:y1,x0:x1];cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA);local=out[y0:y1,x0:x1];local[mk>0]=cand[mk>0]
 return out
def background_surface(obs,mask,sigma=45.):
 vis=(mask==0)&(obs>=200);num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),sigma);den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),sigma);fallback=cv2.GaussianBlur(obs,(0,0),sigma);return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
def support_maps(obs,mask):
 dark=(obs<190)&(mask==0);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0;n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);c=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,hg,a=stats[i]
  if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
 k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8);return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0
def geometry_t80(obs,mask,telea):
 paper=background_surface(obs,mask);norm=cv2.divide(obs,np.maximum(paper,1),scale=255);h,v,c=support_maps(obs,mask);m=mask>0;pres=m&(h|v|c)&(obs<200);rem=m&~pres&(obs>=GEOMETRY_THRESHOLD)&(norm>=GEOMETRY_THRESHOLD);amb=m&~pres&~rem;out=obs.copy();out[rem]=paper[rem];out[amb]=telea[amb];return out,{'preserve':pres,'remove':rem,'ambiguous':amb,'paper':paper,'horizontal':h,'vertical':v,'compact':c}

def load_probability(path,shape):
 if not path:return None
 p=Path(path)
 if p.suffix=='.npy':a=np.load(p)
 elif p.suffix=='.npz':
  with np.load(p) as z:
   key='p_print' if 'p_print' in z.files else z.files[0];a=z[key]
 else:return None
 a=np.asarray(a,np.float32);return a if a.shape==shape else None
print('Canonical helpers ready.')

## 4. Build page maps, probability bands and counterfactual A1-plus-paper candidates

In [ ]:
def band_id(p):
 for b in BANDS:
  if b['low']<=p<b['high']:return b['id']
 return 'UNASSIGNED'
def dilate(m,r=REVIEW_RADIUS):
 return cv2.dilate(m.astype(np.uint8),np.ones((2*r+1,2*r+1),np.uint8))>0 if r>0 else m
def blend(gray,layers,alpha=OVERLAY_ALPHA):
 base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR);colour=base.copy();active=np.zeros(gray.shape,bool)
 for m,bgr in layers:colour[m]=bgr;active|=m
 out=base.copy();out[active]=cv2.addWeighted(base,1-alpha,colour,alpha,0)[active];return out
def legend(image,title,items):
 h,w=image.shape[:2];banner=max(70,int(h*.04));canvas=np.full((h+banner,w,3),255,np.uint8);canvas[banner:]=image;scale=max(.42,min(.82,w/1800));cv2.putText(canvas,title,(12,int(banner*.38)),cv2.FONT_HERSHEY_SIMPLEX,scale,(30,30,30),1,cv2.LINE_AA);x=12;y=int(banner*.78)
 for label,bgr in items:
  cv2.rectangle(canvas,(x,y-14),(x+18,y+4),bgr,-1);cv2.putText(canvas,label,(x+24,y+3),cv2.FONT_HERSHEY_SIMPLEX,scale*.7,(30,30,30),1,cv2.LINE_AA);x+=max(130,int(len(label)*8+52))
 return canvas
def heatmap(gray,prob,valid):
 norm=np.clip((prob-HEATMAP_MIN)/(1-HEATMAP_MIN),0,1);colour=cv2.applyColorMap((norm*255).astype(np.uint8),cv2.COLORMAP_TURBO);base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR);out=base.copy();active=valid&(prob>=HEATMAP_MIN);out[active]=cv2.addWeighted(base,1-OVERLAY_ALPHA,colour,OVERLAY_ALPHA,0)[active];return out

page_rows=[];band_rows=[];threshold_rows=[];component_rows=[];joint_rows=[]
for _,r in roster.iterrows():
 key=str(r.page_key);obs=read_gray(r.observed_path);seed=read_gray(r.seed_path);pann=np.load(r.confidence_path).astype(np.float32);require(pann.shape==obs.shape,f'{key}: confidence shape mismatch');pann=np.clip(pann,0,1)
 authority=prepare_mask(seed);telea=telea_reconstruct(obs,authority);v78,routes=geometry_t80(obs,authority,telea);amb=routes['ambiguous'];paper=routes['paper']
 a1=read_gray(r.a1_path) if r.a1_path else v78.copy();a1[amb]=obs[amb]
 require(np.array_equal(a1[amb],obs[amb]),f'{key}: A1 ambiguous mismatch');require(np.array_equal(a1[~(authority>0)],obs[~(authority>0)]),f'{key}: A1 outside authority drift')
 pprint=load_probability(r.p_print_path,obs.shape)
 valid=amb
 d=WHOLE/key;d.mkdir(parents=True,exist_ok=False)
 cv2.imwrite(str(d/'observed.png'),obs);cv2.imwrite(str(d/'a1.png'),a1);cv2.imwrite(str(d/'authority.png'),authority);cv2.imwrite(str(d/'ambiguous.png'),amb.astype(np.uint8)*255)
 hmap=legend(heatmap(obs,pann,valid),f'{key} - YOLO annotation confidence within A1 AMBIGUOUS',[('low',(255,0,0)),('mid',(0,255,0)),('high',(0,0,255))]);cv2.imwrite(str(d/'p_annotation_heatmap.png'),hmap)
 layers=[];items=[]
 for b in BANDS:
  m=valid&(pann>=b['low'])&(pann<b['high']);layers.append((dilate(m),b['bgr']));items.append((b['label'],b['bgr']))
 ov=legend(blend(obs,layers),f'{key} - rainbow annotation-probability bands in A1 AMBIGUOUS',items);cv2.imwrite(str(d/'p_annotation_band_overlay.png'),ov)
 n,labels,stats,_=cv2.connectedComponentsWithStats((authority>0).astype(np.uint8),8)
 for b in BANDS:
  m=valid&(pann>=b['low'])&(pann<b['high']);band_rows.append({'page_key':key,'band_id':b['id'],'band_label':b['label'],'pixels':int(m.sum()),'components':int(len(set(labels[m].tolist())-{0})),'mean_observed':float(obs[m].mean()) if m.any() else np.nan,'mean_mask_depth':float(cv2.distanceTransform((authority>0).astype(np.uint8),cv2.DIST_L2,5)[m].mean()) if m.any() else np.nan})
 for t in PAPER_THRESHOLDS:
  candidate_mask=valid&(pann>=t);candidate=a1.copy();candidate[candidate_mask]=paper[candidate_mask]
  require(np.array_equal(candidate[~candidate_mask],a1[~candidate_mask]),f'{key}/{t}: noncandidate A1 regression');require(np.array_equal(candidate[~(authority>0)],obs[~(authority>0)]),f'{key}/{t}: outside authority drift')
  cv2.imwrite(str(d/f'a1_paper_ge_{str(t).replace(".","p")}.png'),candidate)
  locator=legend(blend(obs,[(dilate(candidate_mask),(0,0,255))]),f'{key} - A1 pixels newly routed to paper at p_annotation >= {t:.3f}',[(f'>= {t:.3f}',(0,0,255))]);cv2.imwrite(str(d/f'candidate_paper_ge_{str(t).replace(".","p")}_overlay.png'),locator)
  threshold_rows.append({'page_key':key,'threshold':t,'candidate_pixels':int(candidate_mask.sum()),'fraction_of_ambiguous':float(candidate_mask.sum()/max(amb.sum(),1)),'components_touched':int(len(set(labels[candidate_mask].tolist())-{0})),'changed_bytes_vs_a1':int(np.sum(candidate!=a1))})
 for cid in sorted(set(labels[valid].tolist())-{0}):
  cm=labels==cid;q=cm&valid;component_rows.append({'page_key':key,'component_id':int(cid),'authority_area':int(cm.sum()),'ambiguous_pixels':int(q.sum()),'p_annotation_mean':float(pann[q].mean()),'p_annotation_max':float(pann[q].max()),**{f'pixels_ge_{str(t).replace(".","p")}':int(np.sum(q&(pann>=t))) for t in PAPER_THRESHOLDS}})
 if pprint is not None:
  np.save(d/'p_print.npy',pprint)
  qlayers=[];qitems=[];defs=[('cleanup candidate',(0,220,0),valid&(pann>=.95)&np.isfinite(pprint)&(pprint<.25)),('conflict',(255,0,255),valid&(pann>=.95)&np.isfinite(pprint)&(pprint>=.50)),('print-like',(255,255,0),valid&(pann<.95)&np.isfinite(pprint)&(pprint>=.50)),('uncertain',(0,191,255),valid&~((pann>=.95)&(pprint<.25))&~((pann>=.95)&(pprint>=.50))&~((pann<.95)&(pprint>=.50)))]
  for label,c,m in defs:qlayers.append((dilate(m),c));qitems.append((label,c));joint_rows.append({'page_key':key,'quadrant':label,'pixels':int(m.sum())})
  cv2.imwrite(str(d/'joint_probability_quadrants.png'),legend(blend(obs,qlayers),f'{key} - joint p_annotation x p_print diagnostic',qitems))
  for ta in [.995,.990,.975,.950,.925,.900,.850,.800]:
   for cp in PPRINT_CEILINGS:
    m=valid&(pann>=ta)&np.isfinite(pprint)&(pprint<cp);joint_rows.append({'page_key':key,'quadrant':f'pann_ge_{ta:.3f}_pprint_lt_{cp:.2f}','pixels':int(m.sum())})
 page_rows.append({'page_key':key,'ambiguous_pixels':int(amb.sum()),'authority_pixels':int((authority>0).sum()),'confidence_nonzero_ambiguous':int(np.sum(valid&(pann>0))),'p_annotation_mean_ambiguous':float(pann[valid].mean()) if valid.any() else np.nan,'p_annotation_max_ambiguous':float(pann[valid].max()) if valid.any() else np.nan,'p_print_available':pprint is not None})
 print('Built',key)

pd.DataFrame(page_rows).to_csv(TABLES/'page_probability_summary.csv',index=False);pd.DataFrame(band_rows).to_csv(TABLES/'probability_band_by_page.csv',index=False);pd.DataFrame(threshold_rows).to_csv(TABLES/'paper_expansion_by_page.csv',index=False);pd.DataFrame(component_rows).to_csv(TABLES/'component_probability_summary.csv',index=False);pd.DataFrame(joint_rows).to_csv(TABLES/'joint_probability_summary.csv',index=False)

## 5. Aggregate summaries and ranked review queues

In [ ]:
pages=pd.read_csv(TABLES/'page_probability_summary.csv');bands=pd.read_csv(TABLES/'probability_band_by_page.csv');thresholds=pd.read_csv(TABLES/'paper_expansion_by_page.csv')
pooled=bands.groupby(['band_id','band_label'],as_index=False).agg(pixels=('pixels','sum'),pages=('page_key',lambda x:int((bands.loc[x.index,'pixels']>0).sum())),components=('components','sum'));pooled['fraction_of_all_ambiguous']=pooled.pixels/max(pages.ambiguous_pixels.sum(),1);pooled.to_csv(TABLES/'probability_band_pooled.csv',index=False)
threshold_pooled=thresholds.groupby('threshold',as_index=False).agg(candidate_pixels=('candidate_pixels','sum'),pages_touched=('candidate_pixels',lambda x:int((x>0).sum())),components_touched=('components_touched','sum'),changed_bytes_vs_a1=('changed_bytes_vs_a1','sum'));threshold_pooled['fraction_of_ambiguous']=threshold_pooled.candidate_pixels/max(pages.ambiguous_pixels.sum(),1);threshold_pooled.to_csv(TABLES/'paper_expansion_pooled.csv',index=False)
queue=thresholds[thresholds.threshold.isin([.995,.95,.90,.80])].pivot(index='page_key',columns='threshold',values='candidate_pixels').fillna(0).reset_index();queue.columns=['page_key']+[f'pixels_ge_{c}' for c in queue.columns[1:]];queue=queue.merge(pages,on='page_key').sort_values(['pixels_ge_0.9','ambiguous_pixels'],ascending=False);queue.to_csv(TABLES/'visual_review_queue.csv',index=False)
plt.figure(figsize=(10,6));plt.plot(threshold_pooled.threshold,threshold_pooled.candidate_pixels,marker='o');plt.gca().invert_xaxis();plt.xlabel('Minimum p(annotation) routed from A1 observed to paper');plt.ylabel('Candidate pixels across HITL');plt.title('A1 paper-band expansion sensitivity');plt.grid(alpha=.25);plt.tight_layout();plt.savefig(VISUALS/'paper_band_expansion_curve.png',dpi=180);plt.close()
print(pooled.to_string(index=False));print(threshold_pooled.to_string(index=False))

## 6. Rainbow contact sheets

In [ ]:
def make_sheet(order,name,files,max_pages=CONTACT_SHEET_PAGES):
 order=list(order)[:max_pages]
 if not order:return
 fig,axes=plt.subplots(len(order),len(files),figsize=(5*len(files),5.8*len(order)),squeeze=False)
 for r,page in enumerate(order):
  for c,(f,title) in enumerate(files):
   p=WHOLE/page/f;im=cv2.imread(str(p),cv2.IMREAD_UNCHANGED)
   if im is not None and im.ndim==3:im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
   axes[r,c].imshow(im,cmap='gray' if im is not None and im.ndim==2 else None);axes[r,c].set_title(f'{page} - {title}');axes[r,c].axis('off')
 plt.tight_layout();plt.savefig(VISUALS/f'{name}.png',dpi=120,bbox_inches='tight');plt.close(fig)
order=queue.page_key.tolist()
make_sheet(order,'top_hitl_annotation_probability_atlas',[('observed.png','observed'),('a1.png','A1'),('p_annotation_heatmap.png','p(annotation) heatmap'),('p_annotation_band_overlay.png','rainbow bands')])
make_sheet(order,'top_hitl_paper_expansion_0995_0950',[('a1.png','A1'),('a1_paper_ge_0p995.png','paper >=.995'),('a1_paper_ge_0p95.png','paper >=.950'),('candidate_paper_ge_0p95_overlay.png','new paper pixels')])
make_sheet(order,'top_hitl_paper_expansion_0900_0800',[('a1.png','A1'),('a1_paper_ge_0p9.png','paper >=.900'),('a1_paper_ge_0p8.png','paper >=.800'),('p_annotation_band_overlay.png','rainbow bands')])
joint=[p for p in order if (WHOLE/p/'joint_probability_quadrants.png').exists()]
make_sheet(joint,'top_hitl_joint_annotation_print_probability',[('observed.png','observed'),('a1.png','A1'),('p_annotation_band_overlay.png','annotation bands'),('joint_probability_quadrants.png','joint evidence')])
print('Contact sheets:',VISUALS)

## 7. Final manifest and upload bundle

In [ ]:
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'notebook':'85_HITL_Annotation_Probability_Heatmaps_and_Band_Expansion_Atlas','read_only_source_data':True,'promotion_authority':False,'pages':len(roster),'excluded_pages':len(exclusions),'notebook83_source':str(NB83) if NB83 else None,'bands':BANDS,'paper_thresholds':PAPER_THRESHOLDS,'pprint_ceilings':PPRINT_CEILINGS,'interpretation':'Diagnostic only. Counterfactual pages are not production outputs. High annotation confidence cannot override correction safety or reviewed authority.'}
(RUN/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))
lines=['NOTEBOOK 85 - HITL ANNOTATION-PROBABILITY HEATMAPS AND BAND ATLAS','',f'Run: {RUN}',f'Complete reviewed pages: {len(roster)}',f'Excluded/incomplete pages: {len(exclusions)}',f'Pages with p_print available: {int(pd.read_csv(TABLES/"page_probability_summary.csv").p_print_available.sum())}','','This notebook does not change the production policy. It visualises and quantifies how an A1-based high-confidence annotation-to-paper band would expand.','Review the rainbow overlays and counterfactual A1 pages before selecting any threshold for paired-truth testing.']
(RUN/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines))
artefacts=[RUN/'executive_summary.txt',RUN/'experiment_identity.json',TABLES/'notebook85_input_roster.csv',TABLES/'notebook85_input_exclusions.csv',TABLES/'page_probability_summary.csv',TABLES/'probability_band_pooled.csv',TABLES/'probability_band_by_page.csv',TABLES/'paper_expansion_pooled.csv',TABLES/'paper_expansion_by_page.csv',TABLES/'component_probability_summary.csv',TABLES/'joint_probability_summary.csv',TABLES/'visual_review_queue.csv',VISUALS/'paper_band_expansion_curve.png',VISUALS/'top_hitl_annotation_probability_atlas.png',VISUALS/'top_hitl_paper_expansion_0995_0950.png',VISUALS/'top_hitl_paper_expansion_0900_0800.png']
if (VISUALS/'top_hitl_joint_annotation_print_probability.png').exists():artefacts.append(VISUALS/'top_hitl_joint_annotation_print_probability.png')
for p in artefacts:
 require(p.exists(),f'Missing upload artefact {p}');shutil.copy2(p,UPLOAD/p.name)
manifest=[]
for p in sorted(UPLOAD.iterdir()):
 if p.is_file():manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False);shutil.make_archive(str(RUN/'notebook85_upload_bundle'),'zip',UPLOAD);print('Bundle:',RUN/'notebook85_upload_bundle.zip')

## Interpretation discipline

The first visual decision is whether the incremental bands principally correspond to residual handwriting or cross printed notation. Do not select a production threshold from HITL appearance alone. Any promising band should next be measured on paired truth, with genuine-print damage and annotation cleanup reported separately and A1 as the comparator.